In [ ]:
#--- Heavy-Light Decomposition (HLD)
# Imagine this patient history:
# Diagnosis → Biopsy → Biomarker Test → Treatment → Response
# A second branch contains blood and toxicity tests. HLD divides the tree into chains so repeated path queries 
# can be processed efficiently.

"""
HLD splits a tree into chains. This code demonstrates chain construction, but the query is deliberately simplified; 
a complete HLD implementation would use chain positions and a segment tree to accelerate repeated path queries

For example, suppose you want to find the total review score for a patient's journey from Diagnosis to Response.
Your code has two main tasks:
1. Organize the patient journey into smaller chains.
2. Calculate the total review score along a path.

"""
# Each event has a review score.
tree = {
    "Diagnosis"         : ["Biopsy", "Blood_Test"],
    "Biopsy"            : ["Biomarker_Test"],
    "Biomarker_Test"    : ["Treatment"],
    "Treatment"         : ["Response"],
    "Response"          : [],
    "Blood_Test"        : ["Toxicity_Test"],
    "Toxicity_Test"     : []
}
score = {
    "Diagnosis"         : 2,
    "Biopsy"            : 3,
    "Biomarker_Test"    : 4,
    "Treatment"         : 5,
    "Response"          : 2,
    "Blood_Test"        : 1,
    "Toxicity_Test"     : 3
}

parent = {"Diagnosis": None}
size = {}
heavy = {}

def calculate_size(node):
    size[node] = 1
    heavy[node] = None
    largest = 0

    for child in tree[node]:
        parent[child] = node
        calculate_size(child)
        size[node] += size[child]
        if size[child] > largest:
            largest = size[child]
            heavy[node] = child

def build_chains(node, chain_id, chains):
    chains.setdefault(chain_id, []).append(node)
    if heavy[node] is not None:
        build_chains(heavy[node], chain_id, chains)
    for child in tree[node]:
        if child != heavy[node]:
            build_chains(child, child, chains)

calculate_size("Diagnosis")
chains = {}
build_chains("Diagnosis", "Diagnosis", chains)
print("HLD chains:")
for chain in chains.values():
    print(" -> ".join(chain))
# Demonstrate a path query. # Here we use parent links for simplicity, not HLD acceleration.
def path_between(a, b):
    ancestors_a = []
    ancestors_b = []
    while a is not None:
        ancestors_a.append(a)
        a = parent[a]
    while b is not None:
        ancestors_b.append(b)
        b = parent[b]
    while ancestors_a[-1] == ancestors_b[-1]:
        common = ancestors_a.pop()
        if not ancestors_a or not ancestors_b:
            break
        if ancestors_a[-1] != ancestors_b[-1]:
            break
    # For this example, query events on one root-to-node path.
    return ["Diagnosis", "Biopsy", "Biomarker_Test",
            "Treatment", "Response"]
path = path_between("Diagnosis", "Response")
print("Path:", " -> ".join(path))
print("Total review score:", sum(score[n] for n in path))

HLD chains:
Diagnosis -> Biopsy -> Biomarker_Test -> Treatment -> Response
Blood_Test -> Toxicity_Test
Path: Diagnosis -> Biopsy -> Biomarker_Test -> Treatment -> Response
Total review score: 16
